# **SIMT编程实践：Transpose算子优化**

## 概述

本小节介绍SIMT编程模式下的Transpose（矩阵转置）算子开发。Transpose用于交换二维矩阵的行和列。

本节按照完整的逐步优化思路展开，逐一验证每一步优化的实际收益：

1. **直接基于GM读写实现Transpose**：暴露转置写地址不连续、以及Thread Block超发带来的性能问题。
2. **限制Thread Block数为物理核数**：仍然直接读写GM，先单独解决Thread Block超发问题，衡量这一步单独带来的收益。
3. **引入UB中转**：把非连续访问转移到UB内部，重点对比两种不同的Thread Block内部处理方式（固定物理核数+线程数为2048、固定物理核数+线程数为1024），说明寄存器溢出会带来什么开销。
4. **优化UB Bank冲突**：通过UB padding消除转置读阶段的bank冲突。
5. **双缓冲流水并行**：让相邻迭代的GM访问与UB内部转置重叠执行；并在这一步之后补充一个纯连续读写的GM带宽基线，衡量当前硬件在不做任何转置计算时的访存效率上限，作为整条优化路径的参照。

每一步都会给出真实编译运行后采集到的性能数据，用于验证该优化点是否达到了预期效果。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 已学习《SIMT编程模型》中核函数、线程索引、线程组织、内存层级等内容。
- 已学习Gather算子编程实战，了解 `blockIdx`、`threadIdx`、`blockDim`、`gridDim` 的基本使用方式。
- 了解基本的Ascend C SIMT算子开发和执行流程。

### 学习目标

完成本小节后，开发者应能够：

- 理解GM直接转置为什么会出现连续读、非连续写的访存模式，并知道如何用一个纯连续拷贝的基线核函数衡量GM带宽上限。
- 学会如何通过引入UB中转优化Transpose的GM访问模式，使全局内存读写保持连续访问。
- 理解Thread Block启动数量与硬件物理核数之间的关系：既要避免Thread Block超发导致的调度排队开销，也要避免单个Thread Block处理过多线程导致的寄存器溢出。
- 掌握UB bank的基本排布方式，以及如何通过padding缓解bank冲突。
- 理解双缓冲（ping/pong）如何让GM访问与UB内部计算相互重叠，进一步压缩执行耗时。
- 能够使用 `msOpProf` 采集并解读每一步优化对应的真实性能数据。

### 本节内容

- 环境准备
- Transpose算子功能介绍
- Transpose算子实现
- 小结


## 1. 环境准备

正式开始学习之前，先执行下方脚本检查CANN Toolkit是否可用，并把CANN环境变量加载到当前Jupyter进程，保证后续能够正常导入相关代码并使用bisheng编译器完成算子的开发与编译。

本节所有编译、运行和练习修改都在 `Sources/07.05` 目录下进行，`src` 目录仅作为只读的源码仓库存放原始代码。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/07.05")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. Transpose算子功能介绍

Transpose用于交换二维矩阵的行和列，计算公式如下：

```text
output(col, row) = input(row, col)
```

实际计算中，`input` 和 `output` 都按一维数组存储。设输入矩阵形状为 `height x width`，对于输入矩阵中的 `input(row, col)`，其一维下标为：

```cpp
input_index = row * width + col;
```

转置后，该元素写入输出矩阵的 `output(col, row)`，对应的一维下标为：

```cpp
output_index = col * height + row;
```

下图以 `4 x 3` 输入矩阵为例，展示Transpose后行列维度和元素排列的变化：

![](images/07_05_simt_transpose/transpose_example.png)

本节实现中，输入矩阵的 `height` 和 `width` 均为 `1024`。从最朴素实现到后续一系列性能优化路径如下所示：

| 核函数名 | 说明 |
| --- | --- |
| `transpose_gm_custom` | 直接基于GM读写实现转置，每个线程处理一个元素 |
| `transpose_gm_core_limited_custom` | 在 `transpose_gm_custom` 基础上增加核内循环，Thread Block数固定为物理核数 |
| `transpose_ub_2tile_core_limited_custom` | UB中转，Thread Block数固定为物理核数，每个Thread Block处理两个tile |
| `transpose_ub_fixed64_custom` | UB中转，Thread Block数固定为物理核数，每个Thread Block每次迭代处理一个tile |
| `transpose_ub_padding_custom` | UB中转 + padding，消除转置读的bank冲突 |
| `transpose_ub_padding_db_custom` | UB中转 + padding + 双缓冲 |

下面，我们将一一对以上实现展开学习。

## 3. Transpose算子实现

### 3.1 准备源码工作目录

完成环境准备并明确Transpose算子功能后，我们开始动手实践。本节按照完整的优化路径，依次实现下面几个版本，分别放在独立目录中：

- `gm`：直接基于GM读写实现Transpose
- `gm_core_limited`：在 `gm` 的基础上限制Thread Block数为物理核数
- `ub_2tile_core_limited`：UB中转，Thread Block数固定为物理核数，每个Thread Block处理两个tile
- `ub`：UB中转，Thread Block数固定为物理核数，每个Thread Block每次迭代处理一个tile
- `ub_padding`：在 `ub` 的基础上引入UB padding，消除bank冲突
- `ub_padding_db`：在 `ub_padding` 的基础上引入双缓冲
- `copy_baseline`：GM带宽基线（纯连续拷贝），在双缓冲实现之后作为额外对比给出

上述版本的完整源码都已经准备在只读目录 `src/07_05_simt_transpose/` 下。执行下面的单元格，把这些源码拷贝到 `Sources/07.05/simt_transpose/` 作为本节的工作目录，后续的编译、运行和修改都只发生在 `Sources` 下：


In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/07_05_simt_transpose")            # 只读源码目录
DST_ROOT = Path("Sources/07.05/simt_transpose")  # 工作目录

VERSIONS = [
    "gm",
    "gm_core_limited",
    "ub_2tile_core_limited",
    "ub",
    "ub_padding",
    "ub_padding_db",
    "copy_baseline",
]

# 清理旧的工作目录，保证每次都从 src 拷贝出干净的一份
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for version in VERSIONS:
    dst = DST_ROOT / version
    dst.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.asc", "*.h", "CMakeLists.txt"):
        for f in sorted((SRC_ROOT / version).glob(pattern)):
            shutil.copy2(f, dst / f.name)
    print(f"{version}: {sorted(p.name for p in dst.iterdir())}")


### 3.2 直接基于GM读写实现Transpose


#### 3.2.1 实现思路与代码

作为最简单的实现，本节让每个线程只处理输入矩阵中的一个元素，每个线程根据自己的全局下标 `idx` 直接算出在输入矩阵中的行列坐标，再根据转置映射计算输出地址并写入结果：

```cpp
uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
uint32_t row = idx / width;
uint32_t col = idx - row * width;
// input(row,col) -> output(col,row)
output[col * height + row] = input[idx];
```


输入矩阵共有 `1024 * 1024` 个元素，由于计算复杂度低，每个Thread Block可启动 `2048` 个线程，让线程尽量并行起来，因此需要启动的Thread Block数为：

```cpp
uint32_t num_blocks = input_total_length / THREAD_COUNT; // 1024 * 1024 / 2048 = 512
```

对应的启动配置如下：

```cpp
dim3 grid(512, 1, 1);
dim3 block(2048, 1, 1);
```

完整实现代码已保存在 `Sources/07.05/simt_transpose/gm/transpose_gm.asc`，执行下面的单元格查看完整源码：

In [ ]:
!cat Sources/07.05/simt_transpose/gm/transpose_gm.asc

#### 3.2.2 CMake配置

对应的 `CMakeLists.txt` 内容如下(源码文件为`Sources/07.05/simt_transpose/gm/CMakeLists.txt`)：


```cmake
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

find_package(ASC REQUIRED)
project(simt_transpose_gm LANGUAGES ASC CXX)

add_executable(demo
    transpose_gm.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)
```

**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定NPU架构版本，`dav-` 后为架构号，Ascend 950PR/Ascend 950DT 对应 `dav-3510` |
| `--enable-simt` | **启用SIMT编程场景**，编译SIMT算子必须添加 |


#### 3.2.3 编译运行并采集性能

执行以下命令编译并运行当前实现：


In [ ]:
!cd Sources/07.05/simt_transpose/gm && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：


In [ ]:
!cd Sources/07.05/simt_transpose/gm/build && msopprof ./demo

本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写GM（Thread Block超发） | `transpose_gm_custom` | 57.37 | 512 | 6.12 |

`aiv_vec_time` 只有6.12μs，但Task Duration却高达57.37μs，`aiv_vec_time`统计的是每个线程块的Vec操作耗时，两者差异大的主要原因在于：本实现按"一个线程处理一个元素"的最简单方式启动，Thread Block数直接等于 `总元素数 / 每Block线程数 = 512`，远超设备物理核数（64），因此一个物理核平均要串行处理512/64 = 8个线程块任务。

`aiv_vec_time`统计的是每个线程块的平均AIV耗时，那么每个物理核实际的端到端`aiv_vec_time`约等于6.12μs * 8 = 48.96μs。超发的Thread Block需要排队等待前面的执行完成才能调度，每次调度都会产生额外的头尾开销，从上述数据大约可计算出整体头尾开销约为57.37 - 48.96 = 8.4μs，说明这种质朴写法虽然简单，但确引入较重的头尾开销，性能差。

从当前数据看，启动512个线程块引入的开销极大，下面优先解决超发问题。

### 3.3 限制Thread Block数为物理核数

#### 3.3.1 实现思路与代码

本实现将在上一实现的基础上优化线程块数量，实现很简单：保持 `transpose_gm_custom` 直接读写GM的坐标计算不变，只是不再让Thread Block数直接等于总元素数除以每Block线程数，而是限制在物理核数，核内增加一层grid-stride for循环，让每个Thread Block循环处理多组元素：

```cpp
uint32_t stride = gridDim.x * blockDim.x;
for (uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x; idx < total_elements; idx += stride) {
    uint32_t row = idx / width;
    uint32_t col = idx - row * width;
    output[col * height + row] = input[idx];
}
```

封装 `get_vector_core_num()` 接口，在host侧通过 `aclrtSetDevice` 接口获取当前设备的实际物理核数。封装的 `get_vector_core_num()`实现如下：

```cpp
// 查询硬件 vector core 数（AIV 核数），不同环境上的物理核数不同，需要在运行时获取
uint32_t get_vector_core_num(uint32_t device_id)
{
    int64_t core_num = 0;
    aclError ret = aclrtGetDeviceInfo(device_id, ACL_DEV_ATTR_VECTOR_CORE_NUM, &core_num);
    if (ret != ACL_SUCCESS) {
        return 0;
    }
    return static_cast<uint32_t>(core_num);
}
```


启动配置从 `dim3 grid(512, 1, 1)` 改为按查询结果设置：

```cpp
uint32_t num_blocks = get_vector_core_num(device_id);
dim3 grid(num_blocks, 1, 1);
dim3 block(2048, 1, 1);
```

这一步没有改变任何GM访问坐标，转置写回依然是跨行、非连续的；唯一的变化是Thread Block数从512降到物理核数（测试机器上为64），核内循环覆盖原来由多个Thread Block分担的工作量。完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/gm_core_limited/` 目录下（核心逻辑如上），此处不再重复展示全文。

In [ ]:
!cat Sources/07.05/simt_transpose/gm_core_limited/gm_core_limited.asc

#### 3.3.2 编译运行并采集性能

完成CMake配置后，执行以下命令编译并运行当前实现：

In [ ]:
!cd Sources/07.05/simt_transpose/gm_core_limited && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

In [ ]:
!cd Sources/07.05/simt_transpose/gm_core_limited/build && msopprof ./demo

本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写GM（Thread Block超发） | `transpose_gm_custom` | 57.37 | 512 | 6.12 |
| 直接读写GM（限制核数为64） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 32.899 |

`Block Dim` 从512降为64后，Task Duration明显下降（57.37μs → 35.674μs），说明消除Thread Block超发确实带来了收益；但 `aiv_vec_time` 却大幅上升（6.12μs → 32.899μs）。原因是：转置写回阶段跨行写入导致GM写不连续，同一Warp内相邻线程的写地址被分散到输出矩阵的不同行，这部分开销在上一实现中被线程块切换的头尾开销所掩盖，没有体现在 `aiv_vec_time` 耗时上；现在64个Thread Block各自循环处理8组元素，同样的非连续写开销被均匀摊到了每次循环迭代中，因此完整地反映在了 `aiv_vec_time` 里。也就是说，这一步只是把"调度排队开销"转化成了"暴露出来的访存开销"，两者此消彼长，Task Duration的下降幅度小于预期，非连续访问的问题并没有解决。

下一步将尝试引入UB中转，把这部分非连续访问转移到访问效率更高的UB内部，解决GM非连续访问带来的性能损耗。

### 3.4 引入UB中转优化Transpose


#### 3.4.1 UB中转的基本思路

3.3节已经把Thread Block数固定为物理核数，消除了调度排队开销，但没有解决非连续访问本身的问题，因为直接基于GM读写时，输入侧读取是连续的，但转置写回输出矩阵时会沿列方向跨行写入，GM写地址不连续。为了改善写回阶段的访问模式，本实现引入UB作为中间缓存，并把 `1024 x 1024` 输入矩阵划分为多个 `32 x 32` tile。这样可以以tile为单位组织转置：输入矩阵中的一个tile先按行方向连续读入UB，完成tile内转置后，再写回到输出矩阵中对应的tile位置。

这里选择tile块单元大小为 `32 x 32`，主要是因为WarpSize是32，能够保证一个Warp内访存合并，同时便于用tile内的行列坐标描述线程负责的数据位置。对于输入为 `1024 x 1024` 矩阵，行方向和列方向各有 `32` 个tile，因此一共有 `32 * 32 = 1024` 个tile块。

在代码中，所有tile按行优先顺序编号。设 `tiles_per_row = width / tile_dim(32)`，则一维 `tile_id` 与tile的二维坐标关系为：

```cpp
uint32_t tile_row = tile_id / tiles_per_row;
uint32_t tile_col = tile_id - tile_row * tiles_per_row;
```

在单个tile内，使用 `local_row` 表示tile内行号，使用 `local_col` 表示tile内列号。结合tile的二维坐标后，当前元素在输入矩阵中的全局坐标为：

```cpp
uint32_t input_row = tile_row * tile_dim + local_row;
uint32_t input_col = tile_col * tile_dim + local_col;
```

转置后，输入tile `(tile_row, tile_col)` 会写入输出矩阵中的tile `(tile_col, tile_row)`。写回时仍按输出tile的行方向组织访问，其全局坐标关系为：

```cpp
uint32_t output_row = tile_col * tile_dim + local_row;
uint32_t output_col = tile_row * tile_dim + local_col;
```

下图展示了UB中转转置的数据流向。

<img src="./images/07_05_simt_transpose/transpose_ub_dataflow.png" alt="transpose_ub_dataflow"  width="700px" >

从图中可以看到，线程先按输入矩阵原布局把一个tile连续读入UB；同步后，再从UB中按转置方向读取，并按输出矩阵行方向连续写回GM。这样访问模式从 `GM连续读 + GM非连续写` 变为 `GM连续读 + UB转置方向读 + GM连续写`。换句话说，这个优化并没有消除所有不连续访问，而是把不连续访问从GM转移到了UB内部，保障GM的访问都是连续的，因为UB的访问效率较高，非连续访问的性能影响比GM的小很多。

我们在[SIMT内存介绍课程](../03_programming_model/03.04.04_simt_memory_hierarchy.ipynb)中已经学习到，配置的最大线程数决定了每个线程拥有的寄存器个数。引入UB中转后，线程数越多，处理的tile越多，索引变量和分支判断增多，容易导致寄存器溢出；处理太少则会影响并发效率，所以还需要确定每个Thread Block处理启动多少线程数比较合适。下面先尝试启动2048个线程，若存在寄存器溢出问题，再逐步减少线程数。

#### 3.4.2 2048线程版本的核心代码实现

由于线程数是2048，所以每个线程块能完成2个tile块的计算任务：`2048 / (32 * 32) = 2`，为理解方便，我们将这2个tile块组合成一个Tile组（`TILES_PER_BLOCK = 2`）。延续上一实现，把线程块数固定为运行时查询到的硬件物理核数（`get_vector_core_num()`，测试机器上为64）来抑制头开销，因此每个线程块通过循环处理多组tile：

```cpp
constexpr uint32_t TILES_PER_BLOCK = 2;
uint32_t num_blocks = get_vector_core_num(device_id); // 固定为物理核数，运行时查询
dim3 grid(num_blocks, 1, 1);
dim3 block(32, 32 * TILES_PER_BLOCK, 1); // 2048 个线程

for (uint32_t tile_base = blockIdx.x * TILES_PER_BLOCK; tile_base < total_tiles;
     tile_base += gridDim.x * TILES_PER_BLOCK) {
    uint32_t tile_id = tile_base + local_tile;
    // ... 加载、同步、转置写回、同步 ...
}
```

以 `blockIdx.x = 0` 为例，它依次处理tile `(0,1), (128,129), (256,257), ...`；64个Thread Block协同覆盖全部1024个tile，每个Thread Block需要循环8次。循环体内的两次 `asc_syncthreads()` 分别起到不同作用：第一次同步确保Thread Block内所有线程都完成tile数据加载后，才能开始按转置方向读取UB；第二次同步确保当前tile的转置写回全部完成后，才能进入下一轮循环开始加载新数据，避免新一轮加载覆盖还未写出的UB内容。

完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/ub_2tile_core_limited/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.05/simt_transpose/ub_2tile_core_limited/ub_2tile_core_limited.asc

**CMake配置说明：**

由于本实现引入UB后计算复杂度增加不少，需要关注寄存器是否充足，因此 `CMakeLists.txt` 中额外添加了 `--cce-res-usage` 编译选项，用于在编译日志中输出寄存器和栈使用信息。完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/ub_2tile_core_limited/` 目录下。

编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_2tile_core_limited && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


编译日志中会打印类似下面的信息：

```text
[BISHENG] Function properties for _Z38transpose_ub_2tile_core_limited_customILj32EEvPfPKfjjj_simt_entry: Stack size: 24 bytes, Used register number: 16
```

`register number: 16` 说明寄存器已经用满，而 `Stack size: 24 bytes` 说明栈空间使用较多，当前核函数应该出现了寄存器溢出（spill）。

本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写GM（限制核数为64） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 32.899 |
| UB中转（固定物理核数，2 tile/block） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 24.54 |

两者Block Dim相同，都是64，因此可以直接对比引入UB中转本身带来的收益：Task Duration从35.674μs降到27.08μs，`aiv_vec_time` 从32.899μs降到24.54μs，说明把非连续访问转移到UB内部确实降低了访存开销。但编译日志揭示了本实现造成寄存器溢出：每个Thread Block同时维护两个tile需要更多索引变量和分支判断，超出了可用寄存器数量，多余变量换出到栈上，抵消了一部分UB中转带来的收益。回顾前面提到的关系：配置的最大线程数越多，每个线程可用的寄存器就越少，因此下一步尝试降低最大线程数、提升每个线程拥有的寄存器个数，看能否消除寄存器溢出，进一步释放UB中转的潜力。需要注意，"降低线程数换取寄存器"并不是一种通用的优化手段，实际调优时仍需结合具体核函数的计算复杂度，通过实测在两者之间找到平衡点。


#### 3.4.3 降低最大线程数避免寄存器溢出

最大线程数从 `2048` 降为 `1024`：`__launch_bounds__(1024)`, 正好覆盖一个 `32 x 32` tile，每个Thread Block同时维护的tile数从2降为1。启动的核数继续保持为运行时查询到的物理核数，核内循环逐个处理tile：

```cpp
constexpr uint32_t MAX_THREAD_COUNT = 1024;
dim3 grid(num_blocks, 1, 1); // num_blocks = get_vector_core_num(device_id)
dim3 block(32, 32, 1); // 1024 个线程，一次只处理一个 tile

for (uint32_t tile_id = blockIdx.x; tile_id < total_tiles; tile_id += gridDim.x) {
    // ... 加载、同步、转置写回、同步 ...
}
```


核心数据路径如下：


```cpp
uint32_t input_row = tile_row * tile_dim + local_row;
uint32_t input_col = tile_col * tile_dim + local_col;
tile[local_row][local_col] = input[input_row * width + input_col];
asc_syncthreads();

uint32_t output_row = tile_col * tile_dim + local_row;
uint32_t output_col = tile_row * tile_dim + local_col;
output[output_row * height + output_col] = tile[local_col][local_row];
asc_syncthreads();
```

读入UB时，同一个Warp访问输入矩阵同一行的连续32个 `float`，GM读是连续的。写回GM时，同一个Warp写入输出矩阵同一行的连续32个 `float`，GM写也为连续访问。中间的 `tile[local_col][local_row]` 是UB内部的转置方向读取。相比上一步，启动的线程数从2048降为1024，每个线程可用寄存器从16增加到32。而且每个线程块处理的数据量刚好为32*32（1个tile块），索引变量和分支判断更少（不再需要 `local_tile` 区分Thread Block内的两个tile），计算复杂度降低，预期能够消除寄存器溢出。完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/ub/` 目录下。


编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| UB中转（固定物理核数，最大线程数为2048） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 24.54 |
| UB中转（固定物理核数，最大线程数为1024） | `transpose_ub_fixed64_custom` | 25.70 | 64 | 24.01 |

相比上一步，启动的线程数从2048降为1024，编译日志中的Used register number从16变为24、Stack size从24字节降为0字节,寄存器溢出被消除。Task Duration相比上一步继续下降（27.08μs → 25.70μs），`aiv_vec_time` 也小幅下降（24.54μs → 24.01μs），由此可知，引入UB后需要调整最大线程数。正是因为 `1024` 线程正好覆盖一个 `32 x 32` tile，索引关系简单、寄存器压力更低；`2048` 线程需要一个Thread Block同时处理两个tile，会引入更多索引变量、分支和三维UB访问，容易导致寄存器溢出，反而抵消了减少调度开销带来的收益。

不过，`aiv_vec_time` 仍然占了Task Duration的大部分（24.01μs / 25.70μs），说明当前的瓶颈已经从"Thread Block调度"转移到了"UB内部访问效率"。下一步分析并优化UB转置读取阶段的bank冲突。


### 3.5 优化UB Bank冲突

#### 3.5.1 UB bank结构与bank冲突原理

Ascend 950PR/Ascend 950DT 的UB在物理上划分为16个bank，每2个bank组成一个bank group（共8个bank group）；在SIMT编程模式下，每个bank又进一步划分为4个subbank，即整个UB共64个subbank：

<img src="./images/07_05_simt_transpose/bank_structure.png" alt="bank_structure"  width="1500px" >

当同一个Warp内的多个线程在同一条访存指令中，命中同一个bank group内编号相同的subbank时，硬件需要把这些访问串行化处理，由此产生的额外延迟称为bank冲突（更准确地说是subbank冲突）。根据访问类型不同，bank冲突可以分为写写冲突和读读冲突两种。

回顾3.4节的实现，UB中的 `tile` 数组按紧凑的 `32 x 32` 布局存放，每行32个 `float` 共 `32 * sizeof(float) = 128` 字节，恰好跨越4个bank。按照UB的地址低位交织规则，`tile` 的第1行覆盖bank0~bank3，第2行覆盖bank4~bank7，第3行覆盖bank8~bank11，其余行依次类推，每4行回到bank0。转置读取 `tile[local_col][local_row]` 时，同一个Warp内32个线程的 `local_row` 相同、`local_col` 从0到31连续变化，这等价于读取UB tile的同一列，访问的32个地址依次相差128字节。由于行跨度固定为32个 `float`，这32次访问会集中落到两个bank group的subbank 0上，属于读读冲突，如下图所示：

<img src="./images/07_05_simt_transpose/case2_bank.png" alt="case2_bank"  width="1500px" >

要打破这种集中映射，需要改变 `tile` 每行的物理跨度，让同一列的相邻元素错开到不同的subbank上。本节把 `tile` 的行跨度从32增加到34（即每行增加2列padding），每行变为34个 `float`、共 `34 * sizeof(float) = 136` 字节，行跨度也就从16个subbank变为17个subbank。17是奇数，不再与bank/subbank的排布周期对齐，因此同一列的32个元素会依次错开排布到不同的subbank上，同一条访存指令下每个subbank只有一个线程访问，从而消除读读冲突、实现并行读取：

<img src="./images/07_05_simt_transpose/case3_bank.png" alt="case3_bank"  width="1500px" >

由于padding只改变UB内部的物理布局，不改变转置算法本身，也不改变GM读写坐标，因此这个优化只影响UB内部的访问效率，不影响计算结果的正确性。多出来的2列padding不参与计算，只用于让下一行在UB中的起始地址错开。

#### 3.5.2 实现思路与代码

核心差异只有UB数组定义：


```cpp
constexpr uint32_t tile_pad = 2;
constexpr uint32_t tile_pad_stride = tile_dim + tile_pad;
__ubuf__ float tile[tile_dim][tile_pad_stride];
```

有效数据仍写入前32列：

```cpp
tile[local_row][local_col] = input[input_row * width + input_col];
```


转置读和GM写回坐标保持不变：

```cpp
output[output_row * height + output_col] = tile[local_col][local_row];
```

因此这个优化只影响UB内部访问冲突，不影响结果正确性。完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/ub_padding/` 目录下。

编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) |
| --- | --- | --- | --- |
| UB中转（固定物理核数，1 tile/block） | `transpose_ub_fixed64_custom` | 25.70 | 24.01 |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 14.45 |

`aiv_vec_time` 明显下降（24.01μs → 14.45μs），Task Duration也随之下降（25.70μs → 16.11μs），说明padding确实缓解了转置方向读取UB时的bank冲突，SIMT转置读取的效率明显提升。不过 `aiv_vec_time` 仍然占了Task Duration的绝大部分，说明UB内部访问和同步开销依然是主要瓶颈之一。下一步引入双缓冲，让相邻迭代之间的GM访问与UB内部计算重叠执行。

### 3.6 双缓冲流水并行

#### 3.6.1 为什么需要双缓冲：循环尾部同步的代价

`ub_padding` 实现中，每次循环迭代都要经历"加载tile → `asc_syncthreads()` → 转置写回 → `asc_syncthreads()`"，这两次同步分别防止两种数据冲突：

- 第一次同步（加载之后）：确保Thread Block内所有线程都已经把tile数据写入UB，才能开始按转置方向读取，否则可能读到还未写入的数据。
- 第二次同步（写回之后）：确保当前tile的转置读取全部完成，才能进入下一轮循环开始加载新数据，否则下一轮的加载会覆盖还在被读取的UB内容。

第二次同步（循环尾部同步）导致的直接后果是：当前迭代的转置写回必须全部完成后，下一迭代的地址计算和tile加载才能开始，两者之间不存在任何重叠，GM写和GM读被强制串行。
`ub_padding` 实现的仿真指令流水图也揭示了整个流程串行的效果：

<img src="./images/07_05_simt_transpose/case6_trace.png" alt="case6_trace"  width="1500px" >

其中耗时最多的SIMT_LDG和SIMT_STG分别为SIMT编程模式下从GM读取数据和向GM写入数据的指令，每轮循环受尾部 `asc_syncthreads()` 约束，当前轮GM写入阶段完成后才能进入下一轮循环，下一轮数据加载前的地址计算与GM读取阶段需要串行执行。

双缓冲（ping/pong）通过引入两份UB缓冲区来去掉这次尾部同步：用一个在0/1之间切换的下标选择当前使用哪一份缓冲区，当前迭代的转置写回使用其中一份缓冲区的数据，下一迭代的加载则写入另一份缓冲区，不需要等待写回完成即可开始，两者因此可以并行执行。

#### 3.6.2 实现思路与代码

把UB tile数组扩展为两份，并用一个在0/1之间切换的下标 `cnt` 选择当前使用哪一份：

```cpp

__ubuf__ float tile[2][tile_dim][tile_pad_stride];
uint32_t cnt = 0;

for (uint32_t tile_id = blockIdx.x; tile_id < total_tiles; tile_id += gridDim.x) {
    tile[cnt][local_row][local_col] = input[input_row * width + input_col];
    asc_syncthreads(); // 只保留数据加载后的同步

    output[output_row * height + output_col] = tile[cnt][local_col][local_row];

    cnt ^= 1; // 切换到下一份缓冲区
}
```

相比 `ub_padding`，这里去掉了循环尾部（转置写回之后）的 `asc_syncthreads()`，只保留了数据加载之后的同步。加载后的同步不能去掉：转置读取仍然依赖本轮Thread Block内所有线程都已经把数据写入当前这份缓冲区。但去掉尾部同步之后，当前迭代的转置写回和下一迭代的tile加载、地址计算可以在不同的缓冲区上并行进行：下一轮加载写入的是 `tile[cnt ^ 1]`，而当前轮写回读取的是 `tile[cnt]`，两者互不干扰。只有当某一轮真正要读取自己那份缓冲区时（即经过两轮之后再次轮回到同一份缓冲区），才需要等待对应的加载完成——这个等待已经由加载后保留的 `asc_syncthreads()` 隐式保证。

#### 3.6.3 编译运行并采集性能

编译运行双缓冲实现：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding_db && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：


In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding_db/build && msopprof ./demo


本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) |
| --- | --- | --- | --- |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 14.45 |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 10.81 |

去掉循环尾部同步、引入双缓冲后，Task Duration进一步下降（16.11μs → 12.49μs），`aiv_vec_time` 同步下降（14.45μs → 10.81μs）。仿真流水图如下：

<img src="./images/07_05_simt_transpose/case7_trace.png" alt="case7_trace"  width="1500px" >

可以明显看到，相邻迭代之间访存和计算的重叠，当前迭代的转置写回不再阻塞下一迭代的tile加载和地址计算，流水间的耗时掩盖有效减少了整体耗时。

至此，本节的转置优化路径已经全部完成：从最初直接读写GM的实现（57.37μs）优化到双缓冲版本（12.49μs），降到初始版本的约22%。为了衡量这个最终结果距离硬件访存效率的理论上限还有多远，下面补充一个不做任何转置计算的纯连续读写基线，作为整条优化路径的最终参照。

#### 3.6.4 基线对比：GM带宽基线（连续拷贝）

前面的优化路径已经把Task Duration从57.37μs压缩到12.49μs，但这个数值本身并不能说明访存效率是否已经逼近硬件上限。为此，补充一个不做任何转置计算的纯连续拷贝核函数，摸清当前硬件在最理想访存模式下的GM带宽上限，作为整条优化路径的最终参照。这个基线核函数把输入、输出矩阵都当作一维数组，每个线程按线性地址连续读写，不做任何坐标变换：

```cpp
uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
uint32_t stride = gridDim.x * blockDim.x;
for (uint32_t i = idx; i < elements; i += stride) {
    output[i] = input[i];
}
```

这个实现的访存模式是 `GM连续读 + GM连续写`，是所有转置实现理论上能够达到的性能上限参照。如果某个转置实现的Task Duration已经接近这个基线，说明访存效率已经很难再继续提升，后续需要转向计算侧的优化。完整实现代码与CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/copy_baseline/` 目录下。

#### 3.6.5 编译运行并采集性能

CMakeLists.txt已保存在 `Sources/07.05/simt_transpose/copy_baseline/` 目录下，执行以下命令编译并运行当前实现：

In [ ]:
!cd Sources/07.05/simt_transpose/copy_baseline && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：

In [ ]:
!cd Sources/07.05/simt_transpose/copy_baseline/build && msopprof ./demo

本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_scalar_time(μs) |
| --- | --- | --- | --- | --- |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 10.81 | -- |
| GM带宽基线 | `copy_custom` | 6.26 | 4.58 | 0.42 |

这个数值反映了当前硬件在纯连续读写场景下的GM访存效率上限。对比最终的双缓冲实现（12.49μs）和这个基线（6.26μs）可以看到，即便经过完整的优化路径，实际带宽达到理论带宽的50%，也说明如果要继续压缩耗时，需要从减少同步次数、进一步提升数据复用等计算侧手段入手，而不是单纯依赖访存模式优化。

## 4. 小结

本节完整走完了SIMT Transpose算子的逐步优化路径，每一步都通过真实编译运行采集了性能数据，并在最后补充了一个纯连续拷贝的GM带宽基线作为参照：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | 本步引入的变化 |
| --- | --- | --- | --- | --- |
| 直接读写GM | `transpose_gm_custom` | 57.37 | 512 | 每线程处理一个元素，转置写回跨行导致GM写非连续，且Thread Block超发 |
| 直接读写GM（限制核数） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 消除Thread Block超发，但非连续写开销从调度排队中暴露出来 |
| UB中转（固定核数，2 tile/block） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 引入UB中转，GM读写恢复连续，但寄存器溢出 |
| UB中转（固定核数，限制最大线程数） | `transpose_ub_fixed64_custom` | 25.70 | 64 | 降低每Thread Block线程数，消除寄存器溢出 |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 64 | `32x34` padding，消除转置读bank冲突 |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 64 | ping/pong双缓冲，重叠相邻迭代的访存与计算 |

从这条路径中可以总结出几条通用的SIMT算子调优思路：

- **Thread Block数量不要超过物理核数**：按"任务量直接决定Thread Block数"的朴素思路启动核函数（每线程处理一个元素），实现起来最简单，但Thread Block数容易远超物理核数，超发部分需要排队调度，产生额外的头尾开销。
- **优先把非连续访问转移到访问效率更高的存储层级**：直接在GM上非连续读写的问题会造成较大的访存时延，可以考虑引入UB中转，将非连续访问限制在UB内部，GM读写保持连续，能显著提升访存效率。
- **计算复杂度较高时要关注寄存器资源，限制合适的最大线程数避免寄存器溢出**：简单粗暴地把Thread Block数改成物理核数、维持原来的线程数不变，可能因为单个Thread Block要处理的数据和索引变量增多导致寄存器溢出，抵消调度层面的收益。要结合编译日志中的 `Stack size`、`Used register number` 等信息，找到线程数和寄存器压力之间的平衡点。
- **关注片上存储（UB）内部的访问冲突**：即使数据已经搬进UB，转置这种按列访问的模式仍可能因为bank冲突降低访问效率，padding是一种常见且低成本的缓解手段。
- **用双缓冲重叠流水线各阶段**：当循环体内存在"当前迭代结果用完之前，下一迭代无法开始"的强同步点时，双缓冲可以让不同迭代的访存和计算相互重叠，进一步压缩整体耗时。
- **用一个纯连续读写的基线衡量优化空间**：在完成具体算子的优化后，可以用一个不含业务逻辑的纯连续读写核函数摸清硬件访存效率的上限，衡量当前实现距离这个上限还有多远，判断是否还有进一步优化的空间，以及后续应该往访存侧还是计算侧发力。

需要说明的是，本节展示的性能数据全部来自Ascend 950 环境、CANN 9.1.0 版本实测，实际数值会受设备状态、驱动版本等因素影响而波动，但各阶段之间的相对趋势（GM直接转置最慢、UB中转后大幅提升、Thread Block数量与寄存器压力需要权衡、padding和双缓冲进一步优化、纯连续拷贝基线远快于所有转置实现）具有普遍参考意义。

还需要特别说明的是，本节的优化路径全部建立在 `1024 x 1024` 这个**完全对齐**的shape上：`height` 和 `width` 都是tile边长32的整数倍，因此核函数里可以放心地整除取tile数，每个线程也都可以无条件读写GM。真实业务中的shape几乎不会这么友好，非对齐shape会在矩阵边界产生不足一个tile的残块，直接套用本节的核函数会**静默算错而不是崩溃**。下面的课后练习就以 `1021 x 1325` 为例，让你把这条路径的最终实现泛化到任意shape，并衡量功能泛化本身的性能代价。

如果你想进一步在一个访存模式完全不同的算子上综合运用本节和下一节（07.06混合编程）的全部技巧，可以继续学习07.07节的大课程作业——那里以MaxPool算子为题，要求独立完成从朴素实现到最终优化的完整链路，并会得到一些和Transpose截然不同的结论。

## 课后编程习题：非对齐shape的功能泛化

### 习题背景

正文的整条优化路径都建立在一个隐含假设之上：`height` 和 `width` 都是tile边长 `32` 的整数倍。`1024 x 1024` 恰好满足这个条件，于是3.6节的最终实现 `transpose_ub_padding_db_custom` 里可以放心地整除取tile数：

```cpp
uint32_t tiles_per_row = width / tile_dim;   // 1024 / 32 = 32，整除
```

并且每个线程都可以无条件地读写GM：

```cpp
tile[cnt][local_row][local_col] = input[input_row * width + input_col];      // 一定在矩阵内
output[output_row * height + output_col] = tile[cnt][local_col][local_row];  // 一定在矩阵内
```

而真实业务里的shape几乎不会这么友好。本习题请你**在3.6节最终实现（UB padding + 双缓冲）的基础上补充边界判断**，把它泛化成对任意 `height x width` 都能算对的版本，并衡量这层泛化本身带来的性能代价。

**非对齐shape会破坏什么？** 以 `1021 x 1325` 为例：

| 维度 | 长度 | 除以32 | 完整tile数 | 残块 |
| --- | --- | --- | --- | --- |
| height | 1021 | 31.90… | 31 | 最后剩 `1021 - 31*32 = 29` 行 |
| width | 1325 | 41.40… | 41 | 最后剩 `1325 - 41*32 = 13` 列 |

于是矩阵被切成 `32 x 42 = 1344` 个tile，但其中最后一行tile只有29行有效数据，最后一列tile只有13列有效数据，右下角那个tile同时缺行又缺列。沿用正文的核函数会同时踩两个坑：

1. **`width / tile_dim` 向下取整丢块**：`1325 / 32 = 41`，核函数算出 `tiles_per_row = 41`，而host侧真正需要的是 42。`tile_id` 到 `(tile_row, tile_col)` 的换算全部错位，整个结果都是乱的。
2. **边界tile的线程越界访问GM**：即使 `tiles_per_row` 修对，边界tile里仍有部分线程算出的 `(input_row, input_col)` 落在矩阵之外。这些线程会读到相邻行的数据，写回时同样会覆盖本不该写的位置。

### 规格

| 项 | 取值 |
| --- | --- |
| 待泛化的基线 | 正文3.6节 `transpose_ub_padding_db_custom`（UB padding + 双缓冲） |
| 主用例输入 | **(1021, 1325)**，`float`；输出 **(1325, 1021)** |
| tile大小 | `32 x 32`，UB布局 `32 x 34`（padding 2列，与正文一致） |
| Thread Block数 | `min(物理核数, tile总数)` |
| 每Block线程数 | 1024（`dim3(32, 32, 1)`） |

### 要求

在骨架的 `transpose_unaligned_ub_padding_db_custom` 里补全三处 `TODO`：`tiles_per_row` 改向上取整、GM读加边界判断、GM写加边界判断。**UB padding布局、双缓冲的 `cnt` 轮换、转置读的下标 `tile[cnt][local_col][local_row]` 都不需要改动。**

骨架里已经放好了两个核函数：未泛化的对照组（正文3.6节实现的原样拷贝）和待你补全的泛化版本。host侧代码也已写好，支持三种运行模式：

| 命令 | 作用 |
| --- | --- |
| `./demo` | 用例1~9的功能校验 |
| `./demo original` | 用例A：只在 `1024 x 1024` 上启动一次**未泛化**核函数，供 `msopprof` 采集 |
| `./demo generalized` | 用例B：只在 `1024 x 1024` 上启动一次**已泛化**核函数，供 `msopprof` 采集 |

后两种模式各自只启动一次核函数，这样 `msopprof` 采集到的就是指定核函数的Task Duration，不会和其他shape的调用混在一起。

### 功能测试用例

泛化后的核函数必须通过下面9个shape。这些用例不是随手挑的：每一个都针对一类特定的边界情形，合在一起覆盖了"残块出现在哪个方向"、"残块有多大"、"tile数少于物理核数"这几个维度。骨架里的 `cases` 表已经按顺序列好，运行 `./demo` 会逐个校验并打印一行结论。

| # | shape (h x w) | 输出 (w x h) | tile划分 | tile总数 | 启动Block数 | 覆盖的边界情形 |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | **1021 x 1325** | 1325 x 1021 | 32 x 42 | 1344 | 64 | **主用例**：行列均非32倍数，右下角tile同时缺行缺列（有效 `29 x 13`） |
| 2 | 1021 x 1024 | 1024 x 1021 | 32 x 32 | 1024 | 64 | 仅height非对齐：残块只出现在最后一行tile |
| 3 | 1024 x 1325 | 1325 x 1024 | 32 x 42 | 1344 | 64 | 仅width非对齐：残块只出现在最后一列tile |
| 4 | 33 x 31 | 31 x 33 | 2 x 1 | 2 | 2 | 刚跨过一个tile边界：height多1行、width少1列，残块极小 |
| 5 | 32 x 32 | 32 x 32 | 1 x 1 | 1 | 1 | 正好一个tile：无残块，回归验证（骨架也应通过） |
| 6 | 1 x 1325 | 1325 x 1 | 1 x 42 | 42 | 42 | 单行：`height < tile_dim`，每个tile只有1/32的行有效 |
| 7 | 1021 x 1 | 1 x 1021 | 32 x 1 | 32 | 32 | 单列：`width < tile_dim`，此时整除会得到 `tiles_per_row = 0` |
| 8 | 1 x 1 | 1 x 1 | 1 x 1 | 1 | 1 | 极端最小shape：只有1个线程有效 |
| 9 | 1024 x 1024 | 1024 x 1024 | 32 x 32 | 1024 | 64 | 正文主shape：完全对齐，回归验证泛化没有破坏原有功能 |

几点说明：

- **用例5和9是回归用例**：它们完全对齐，补全前后都应该通过。如果泛化后反而在这两个shape上出错，说明边界判断写反了（典型症状：把 `output_row < width` 写成 `output_row < height`，在方阵上恰好看不出来，所以还需要用例1这种非方阵）。
- **用例7和8专门盯 `tiles_per_row = 0`**：`width < 32` 时整除得0，核函数里 `tile_id / tiles_per_row` 就成了除以0。实测这个除法在硬件上**不会触发异常、也不会挂死**，只是算出一个无意义的 `tile_row`，结果静默出错——又一个“不崩只错”的例子。这是只用大shape测不出来的坑。（用例8的 `1 x 1` 在未补全的骨架上反而显示通过，属于巧合：整个矩阵只有1个元素，算错的 `tile_row` 恰好没能让它落到别处。别把这种巧合当成正确性。）
- **用例4、6、7、8的tile总数都小于物理核数（64）**，因此启动的Block数被截断为tile总数，多余的核不参与计算。
- **输入数据全部非零**（`i % 4096 * 1.25 + 1.0`），输出缓冲区在启动前用 `aclrtMemset` 清零，因此结果里残留的 `0` 一定意味着"漏写"，可以和"写错值"区分开。
- **这9个用例同时也是"不泛化会怎样错"的展示**：骨架里的泛化版本还是对照组的原样拷贝，所以第一次运行 `./demo` 得到的失败列表，就是正文3.6节实现直接套到非对齐shape上的真实表现。

### 性能对比用例

功能通过之后，用下面这一组对照采集泛化带来的性能代价。两次采集**用的是同一个shape、同一份输入数据、同一套启动配置**，唯一变量是核函数：

| # | 模式 | 核函数 | shape | Block Dim | 采集方式 |
| --- | --- | --- | --- | --- | --- |
| A | `./demo original` | `transpose_ub_padding_db_custom`（未泛化） | 1024 x 1024 | 64 | `msopprof ./demo original` |
| B | `./demo generalized` | `transpose_unaligned_ub_padding_db_custom`（已泛化） | 1024 x 1024 | 64 | `msopprof ./demo generalized` |

对比指标取 `msopprof` 输出的 **Task Duration(μs)**，辅以 `PipeUtilization.csv` 里的 `aiv_vec_time` / `aiv_scalar_time`，用来判断开销落在哪条流水上。

**关于性能对比的设计**：为什么用**对齐的 `1024 x 1024`** 来测泛化开销，而不是用非对齐的主用例？因为对照组在非对齐shape上根本算不对，拿一个算错的版本比耗时没有意义。而在 `1024 x 1024` 上两个版本**处理的数据量、tile数、循环轮数完全相同，算出的结果也都正确**，唯一的差别就是泛化版本多了取整和两处边界判断——两者的差值才干净地反映"为支持非对齐shape所付出的代价"。


### 准备工作目录

执行下面的单元格，把骨架代码拷贝到工作目录：


In [ ]:
import shutil
from pathlib import Path

SRC = Path("src/07_05_transpose_unaligned_simt")
DST = Path("Sources/07.05/transpose_unaligned")

if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True, exist_ok=True)
for pattern in ("*.asc", "CMakeLists.txt"):
    for f in sorted(SRC.glob(pattern)):
        shutil.copy2(f, DST / f.name)
print(sorted(p.name for p in DST.iterdir()))


### 第一步：先跑一遍骨架，确认非对齐shape如何出错

骨架里的泛化版本目前还是对照组的原样拷贝，因此先直接编译运行，观察失败现象：


In [ ]:
!cd Sources/07.05/transpose_unaligned && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


骨架运行后的输出大致如下（`[FAIL]` 行后面会打印第一个错误元素的位置，便于定位是哪块残块算错了）：

```text
Vector core num: 64

=== 功能校验（用例1~9）===
[FAIL] 1021 x 1325 -> 1325 x 1021 | tiles 32 x 42 = 1344 | blocks 64 | 用例1  主用例：行列均非 32 倍数，右下角 tile 同时缺行缺列
       first mismatch at output index 1021 (row 1, col 0): got 0, expect 2.25
[FAIL] 1021 x 1024 -> 1024 x 1021 | tiles 32 x 32 = 1024 | blocks 64 | 用例2  仅 height 非对齐：残块只在最后一行 tile
       first mismatch at output index 1021 (row 1, col 0): got 0, expect 2.25
[FAIL] 1024 x 1325 -> 1325 x 1024 | tiles 32 x 42 = 1344 | blocks 64 | 用例3  仅 width 非对齐：残块只在最后一列 tile
       first mismatch at output index 1024 (row 1, col 0): got 0, expect 2.25
[FAIL] 33 x 31 -> 31 x 33 | tiles 2 x 1 = 2 | blocks 2 | 用例4  刚跨过一个 tile 边界：残块极小
       first mismatch at output index 0 (row 0, col 0): got 0, expect 1
[ OK ] 32 x 32 -> 32 x 32 | tiles 1 x 1 = 1 | blocks 1 | 用例5  正好一个 tile：无残块，回归验证
[FAIL] 1 x 1325 -> 1325 x 1 | tiles 1 x 42 = 42 | blocks 42 | 用例6  单行：height < tile_dim
       first mismatch at output index 3 (row 3, col 0): got 612.25, expect 4.75
[FAIL] 1021 x 1 -> 1 x 1021 | tiles 32 x 1 = 32 | blocks 32 | 用例7  单列：width < tile_dim，整除会得到 tiles_per_row = 0
       first mismatch at output index 0 (row 0, col 0): got 0, expect 1
[ OK ] 1 x 1 -> 1 x 1 | tiles 1 x 1 = 1 | blocks 1 | 用例8  极端最小 shape：只有 1 个线程有效
[ OK ] 1024 x 1024 -> 1024 x 1024 | tiles 32 x 32 = 1024 | blocks 64 | 用例9  正文主 shape（对齐）：回归验证

[Failed] Case accuracy verification failed!
```

这份输出有几个值得注意的地方：

- **只有回归用例5、9通过**（`32 x 32` 和 `1024 x 1024`，完全对齐），其余非对齐用例全部失败。用例8的 `1 x 1` 显示通过属于巧合。
- **用例2、3说明单边非对齐同样失败**：只要有一个维度不是32的倍数，就一定有残块。
- **没有任何一个用例崩溃或挂死**——全部是"跑完了，但结果是错的"。这是非对齐bug最典型也最危险的症状。
- **用例6的错误值是 `612.25` 而不是0**，尤其值得注意：这不是明显的空洞，而是一个看起来完全合理的浮点数。越界线程读到的是输入矩阵之外的相邻内存值，那里恰好残留着有意义的数据。

现在开始修。查看骨架代码：


In [ ]:
!cat Sources/07.05/transpose_unaligned/transpose_unaligned_simt.asc

### 第二步：补全三处TODO

三处 `TODO` 分别对应非对齐shape带来的三个问题，改动都很小，但每一处都必要：

**TODO 1：`tiles_per_row` 改成向上取整。**

```cpp
uint32_t tiles_per_row = (width + tile_dim - 1) / tile_dim;
```

这一步是为了保证计算出来的tiles块数需要算上行末的残缺块。

**TODO 2：GM读加边界判断。**

```cpp
if (input_row < height && input_col < width) {
    tile[cnt][local_row][local_col] = input[input_row * width + input_col];
}
```

越界的线程什么都不做，它在UB里对应的那个 `tile[cnt][local_row][local_col]` 保持未初始化。

**TODO 3：GM写加边界判断。**

```cpp
if (output_row < width && output_col < height) {
    output[output_row * height + output_col] = tile[cnt][local_col][local_row];
}
```

注意这里的上界是 **`output_row < width`、`output_col < height`**，因为输出矩阵的形状是 `width x height`，行下标的上界自然是 `width`。这是本习题最容易写错的一处，写反了在方阵上看不出问题，在 `1021 x 1325` 这种非方阵上立刻暴露。


**还有一个容易踩的坑：`asc_syncthreads()` 绝对不能放进 `if` 里。** 边界判断只包住GM访问本身，同步必须由Thread Block内全部1024个线程无条件执行。如果写成下面这样：

```cpp
if (input_row < height && input_col < width) {
    tile[cnt][local_row][local_col] = input[input_row * width + input_col];
    asc_syncthreads();   // 错误：越界线程不会执行到这里
}
```

边界tile里越界的线程不参与同步，Thread Block内的线程就会在不同的同步点上等待，导致挂死或读到还未写入的数据。这一条对所有带条件分支的SIMT核函数都适用。

补全后重新编译运行，确认9个shape全部通过：


In [ ]:
!cd Sources/07.05/transpose_unaligned/build && make -j && ./demo


功能全部正确时，用例1~9全部是 `[ OK ]`，最后输出 `[Success] Case accuracy verification passed.`：

```text
Vector core num: 64

=== 功能校验（用例1~9）===
[ OK ] 1021 x 1325 -> 1325 x 1021 | tiles 32 x 42 = 1344 | blocks 64 | 用例1  主用例：行列均非 32 倍数，右下角 tile 同时缺行缺列
[ OK ] 1021 x 1024 -> 1024 x 1021 | tiles 32 x 32 = 1024 | blocks 64 | 用例2  仅 height 非对齐：残块只在最后一行 tile
[ OK ] 1024 x 1325 -> 1325 x 1024 | tiles 32 x 42 = 1344 | blocks 64 | 用例3  仅 width 非对齐：残块只在最后一列 tile
[ OK ] 33 x 31 -> 31 x 33 | tiles 2 x 1 = 2 | blocks 2 | 用例4  刚跨过一个 tile 边界：残块极小
[ OK ] 32 x 32 -> 32 x 32 | tiles 1 x 1 = 1 | blocks 1 | 用例5  正好一个 tile：无残块，回归验证
[ OK ] 1 x 1325 -> 1325 x 1 | tiles 1 x 42 = 42 | blocks 42 | 用例6  单行：height < tile_dim
[ OK ] 1021 x 1 -> 1 x 1021 | tiles 32 x 1 = 32 | blocks 32 | 用例7  单列：width < tile_dim，整除会得到 tiles_per_row = 0
[ OK ] 1 x 1 -> 1 x 1 | tiles 1 x 1 = 1 | blocks 1 | 用例8  极端最小 shape：只有 1 个线程有效
[ OK ] 1024 x 1024 -> 1024 x 1024 | tiles 32 x 32 = 1024 | blocks 64 | 用例9  正文主 shape（对齐）：回归验证

[Success] Case accuracy verification passed.
```

编译日志中的资源信息也值得看一眼：

```text
[BISHENG] ..._Z30transpose_ub_padding_db_custom..._simt_entry: Stack size: 0 bytes, Used register number: 25
[BISHENG] ..._Z40transpose_unaligned_ub_padding_db_custom..._simt_entry: Stack size: 0 bytes, Used register number: 25
```

两个版本的 `Used register number` 都是25、`Stack size` 都是0，说明**边界判断没有增加寄存器压力、也没有引入溢出**。这是个好消息：泛化的代价停留在指令数量上，没有恶化到寄存器层面，因此正文3.4节那套"降低线程数换寄存器"的调优结论在泛化后依然成立，不需要重新平衡。

### 第三步：用msopprof对比两个版本的性能（用例A / 用例B）

分别采集两个核函数在 `1024 x 1024` 上的性能数据。先采集用例A（未泛化的对照组）：


In [ ]:
!cd Sources/07.05/transpose_unaligned/build && msopprof ./demo original


再采集用例B（已泛化的版本）：

In [ ]:
!cd Sources/07.05/transpose_unaligned/build && msopprof ./demo generalized


### 实测结果

把两次采集的数据填入下表：

| 用例 | 版本 | `tiles_per_row` | GM访问 | Task Duration(μs) |
| --- | --- | --- | --- | --- |
| A | 未泛化（正文3.6节实现） | `width / tile_dim` | 无边界判断 | |
| B | 已泛化（本习题） | `(width + tile_dim - 1) / tile_dim` | 读写各一次边界判断 | |

本机实测（`1024 x 1024`，3次采集取均值，Block Dim均为64）：

| 用例 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_scalar_time(μs) | aiv_total_cycles |
| --- | --- | --- | --- | --- | --- |
| A 未泛化 | `transpose_ub_padding_db_custom` | **23.64** | 10.86 | 0.56 | 18852 |
| B 已泛化 | `transpose_unaligned_ub_padding_db_custom` | **25.72** | 12.07 | 0.40 | 20600 |
| B − A | — | **+2.08（+8.8%）** | +1.21 | −0.16 | +1748 |

### 结论：泛化的代价是多少，代价来自哪里

**1. 功能泛化不是免费的，在这个算子上约8.8%。** 注意这个数字是在**完全对齐**的 `1024 x 1024` 上测出来的：这里一个残块都没有，所有边界判断的结果都是"通过"，泛化版本做的有效工作和对照组一字不差。也就是说，这8.8%是**纯粹为"支持非对齐shape的可能性"付出的固定开销**，与实际是否存在残块无关。

**2. 寄存器用量不变（25 → 25），`Stack size` 均为0。** 泛化的代价停留在指令数量上，没有恶化到寄存器压力层面。

**3. 真实的非对齐shape还要额外付Warp利用率的代价。** 上面8.8%只是判断指令本身。在 `1021 x 1325` 上还有第二重损失：1344个tile里有 `1344 - 31*41 = 73` 个是残块，以最后一列tile为例，只有13/32的线程真正搬数据，其余19/32的线程在这条访存指令上完全空转——但它们仍然占着Warp的发射槽位，也仍然要参与 `asc_syncthreads()`。同一个Warp内只要有线程被判断掉，这次访存的有效带宽就等比例下降。

所以完整的代价是两层：**固定的判断指令开销（对齐shape上也要付，约8.8%）+ 残块上的Warp利用率损失（只在真正非对齐时出现）**。`1 x 1325` 这类极端shape是后者的放大版：42个tile里每个都只有 `1 x 32` 有效，1024个线程里只有32个在干活，利用率仅3%。真实算子开发中遇到这种shape，正确做法是换一种tile划分（比如把tile退化成一维），而不是硬套 `32 x 32` 的方案。

### 思考题

1. 为什么 `tiles_per_row` 在核函数里必须和host侧 `total_tiles` 用同一个取整公式？如果核函数用 `width / tile_dim`（整除）而host侧用向上取整，`tile_id = 41` 会被换算成哪个 `(tile_row, tile_col)`？（提示：`width = 1325` 时核函数算出 `tiles_per_row = 41`，代入算一算。）

2. 写回判断为什么是 `output_row < width && output_col < height`，而不是 `output_row < height && output_col < width`？如果写反了，在 `1024 x 1024` 这种方阵上能发现吗？在 `1021 x 1325` 上会在哪个元素上首先暴露？

3. 骨架在用例6（`1 x 1325`）上读到的错误值是 `612.25` 而不是0。`height = 1` 时只有 `local_row = 0` 的那32个线程落在矩阵内，其余992个线程算出的 `input_row >= 1`，它们的 `input_row * width + input_col` 已经超出了 `input` 这块只有1325个float的内存。请回答：这些越界线程读到的是什么？为什么读出来不是0、也没有触发地址异常？（提示：`aclrtMalloc` 分配的实际页面通常大于请求的字节数，相邻地址上可能残留着之前分配、释放过的数据。）再想一想，这类"跑完了但结果是错的、且错误值看起来很合理"的bug，为什么比直接崩溃更难发现？

4. 除了本习题采用的"边界判断法"，处理非对齐shape还有一种常见思路：**把输入矩阵padding到32的倍数**（申请一块 `1024 x 1344` 的内存，多余部分填0，转置后只拷回有效区域）。这样核函数可以完全不加边界判断，省掉那8.8%。请从内存开销、host侧拷贝次数、核函数复杂度三个角度对比这两种方案，说明各自适合什么场景。

**本习题的核心结论**：正文那条优化路径的每一步都建立在"shape对齐"这个隐含假设上，而真实算子必须先做到功能泛化，才谈得上性能。泛化的代价是可以量化的——在同一个对齐shape上对比泛化前后，本例约8.8%，不会恶化寄存器压力；真正的非对齐shape上还要再叠加残块处的Warp利用率损失。另外，非对齐bug的典型症状是**静默算错而不是崩溃**，因此边界shape必须进测试用例，不能只测对齐shape。

**参考答案：**


In [ ]:
!cat answer/07_05_transpose_unaligned_ub_padding_db/transpose_unaligned_simt.asc